# Laptop lab 1: Visualization
*Look at your data before you model it*

**Machine Learning with Python** · Minor Amsterdam Data Science and Artificial Intelligence · Fall 2026<br>
Thursday, October 29, 2026 · Laptop labs: Batuhan Ataş (b.atas@uva.nl)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/batuatas/MLwPython-labs/blob/main/lab01.ipynb)

> **Before you start:** choose **File → Save a copy in Drive** and work in that copy. Colab does not save your changes in the original notebook.

In Lecture 2 you saw how to plot data and what plots can reveal: Anscombe's four datasets, numbers stored as text, values that mean different things, suspicious round numbers. The lecture used a small, synthetic student survey. Today you meet real data, with all their problems, and nobody tells you in advance where the problems are.

**The story.** A friend of yours wants to rent out her apartment in Amsterdam on Airbnb, and asks you: *what can I ask per night?* You have a table with the Amsterdam listings on Airbnb, collected from the Airbnb website on September 16, 2026: more than 10,000 of them. Before you answer her, or let a model answer her, you look carefully at the data.

Three questions guide you through the lab:

1. **Can we trust the data?** Tasks 1 to 3: a first look, missing prices, and listings that stand out.
2. **What goes with the price?** Tasks 4 and 5: a map of the city, and the features that matter.
3. **What changes over time?** Task 6: sixteen years of reviews.

Then you ask the data a question of your own (Task 7), and you test yourself with a few exam-style questions.

## The data

[Inside Airbnb](https://insideairbnb.com) collects the public listings of Airbnb in many cities. We use two files of Amsterdam:

- `airbnb_listings.csv`: one row per listing, 20 of the original columns.
- `airbnb_reviews_daily.csv`: how many reviews guests wrote on each day, from 2010 to 2026. A review is written after a stay, so it tells you roughly how busy Airbnb was.

The codebook of the listings:

| Column | Meaning |
|---|---|
| `id` | number of the listing |
| `neighbourhood` | one of the 22 neighborhoods of Amsterdam (the data use British spelling) |
| `room_type` | `Entire home/apt`, `Private room`, `Hotel room` or `Shared room` |
| `property_type` | as the host describes it, for example `Entire rental unit` or `Boat` |
| `accommodates` | maximum number of guests |
| `bedrooms`, `beds` | number of bedrooms and beds |
| `price` | price per night **in euros**; the `$` sign is a technical artifact of the export (Inside Airbnb's data dictionary) |
| `minimum_nights`, `maximum_nights` | shortest and longest stay the host allows |
| `availability_365` | number of nights that can still be booked in the coming 365 days |
| `number_of_reviews` | number of reviews the listing has received |
| `first_review`, `last_review` | dates of the first and the latest review |
| `review_scores_rating` | average rating by the guests, at most 5 |
| `host_is_superhost` | `t` (true) or `f` (false): Airbnb's label for experienced, highly rated hosts |
| `host_listings` | number of listings this host has in Amsterdam |
| `license` | the registration number that the city requires |
| `latitude`, `longitude` | location of the listing |

## How this lab works

The notebook mixes explanations and code. You will meet these kinds of cells:

| You see | What you do |
|---|---|
| **Example** | Run the cell and read the code. It shows you a new idea before you use it yourself. |
| **Your turn** | Replace the `...` with your own code, then run the cell. |
| **Check** | Run it right after your own cell. It says ✓ when your answer is right, or tells you what to look at. |
| **Hint** | Click to unfold it when you are stuck. Try for a few minutes first. |
| **Think** | Double-click *Your answer* and write one or two sentences. We discuss some of the answers together. |

The *Your turn* cells give you less code than the examples of Lecture 2: often you have to find out how yourself. That is on purpose. The tables of commands in Task 1 and the hints are there to help you. Run the cells from top to bottom with **Shift + Enter**: later cells use the variables of earlier ones. Stuck for more than five minutes? Raise your hand: that is what the lab is for.

| Task | Minutes |
|---|---|
| 0. Setup | 5 |
| 1. A first look | 10 |
| 2. Prices, and prices that are missing | 12 |
| 3. Listings that stand out | 15 |
| 4. A map of the city, in three colormaps | 12 |
| 5. What goes with the price? | 20 |
| 6. Sixteen years of reviews | 8 |
| 7. Your own question | 8 |
| Check your understanding | together, at the end |
| If you finish early | at home |

You will not finish everything in the lab, and that is fine: you can finish it at home. Is Python new for you? Every *Your turn* has a hint that shows the pattern you need, and the [pandas tutorials](https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html) 1 to 3 are a good start.

## Task 0. Setup

Run the cell below. It loads the Python libraries you need today, downloads the checks of this lab, and tells you whether everything works. You should see a green box with a ✓.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# The data and the checks of this lab are in the course lab repository.
BASE = "file:///Users/batas/ForBatuhan/MLwPython-labs/"import urllib.request
urllib.request.urlretrieve(BASE + "lab01_tools.py", "lab01_tools.py")
from lab01_tools import *

LISTINGS = BASE + "data/airbnb_listings.csv"
REVIEWS = BASE + "data/airbnb_reviews_daily.csv"

setup_ok()

: 

## Task 1. A first look

**Example.** Load the listings and show the first five rows.

In [ ]:
listings = pd.read_csv(LISTINGS)
listings.head()

Lecture 2 showed you how to make plots, and four lines to run on every new dataset. Today you also need some pandas commands that the lecture did not show. You find them in this table; come back to it whenever you need one. More commands follow later, each where you first need it.

| Code | Gives |
|---|---|
| `len(listings)` | the number of rows |
| `listings["price"]` | one column |
| `listings["accommodates"].max()` | the largest value of a column (also `.min()`, `.mean()`, `.median()`) |
| `listings["room_type"].nunique()` | the number of different values |
| `listings["room_type"].value_counts()` | how often each value occurs, the most common first |
| `listings["accommodates"] > 4` | `True` or `False` for every listing |
| `(listings["accommodates"] > 4).sum()` | how many listings meet the condition: `True` counts as 1 |
| `(listings["accommodates"] > 4).mean()` | which *share* of the listings meets it, between 0 and 1 |
| `listings[listings["accommodates"] > 4]` | only the rows that meet it |
| `listings.loc[rows, "price"]` | the column `price` of the selected rows only |
| `listings.groupby("room_type")["accommodates"].median()` | the median per group (Lecture 2, bar chart) |
| `listings.sort_values("accommodates")` | the rows sorted, smallest first; add `ascending=False` for largest first |

**Your turn.** Get to know the table: how many listings are there, in how many different neighborhoods, and which room type is the most common? Store your answers in the three variables below. Compute each value with code, instead of typing a number you read from an output: then your code still gives the right answer when the data change.

In [ ]:
n_listings = ...          # a number
n_neighbourhoods = ...    # a number
most_common_room = ...    # a name in quotes, or computed

<details><summary><b>Hint</b> (click to unfold)</summary>

- `len(listings)` counts the rows; `.nunique()` counts different values.
- `listings["room_type"].value_counts()` lists the room types, the most common first. `.index[0]` takes the first name.

</details>

In [ ]:
# Check
check_1()

**Your turn.** Now run the four lines of Lecture 2, "Four lines before any plot", on `listings`: the types of the columns, the missing values per column, the number of duplicate rows, and the summary statistics. Use one cell per line, so that you can read each output on its own. Look carefully at every output: you will need what you see in the next tasks.

In [ ]:
# types of the columns
...

In [ ]:
# missing values per column
...

In [ ]:
# duplicate rows
...

In [ ]:
# summary statistics
...

<details><summary><b>Hint</b> (click to unfold)</summary>

```python
listings.dtypes              # types per column
listings.isna().sum()        # missing values
listings.duplicated().sum()  # duplicate rows
listings.describe()          # min, max, mean, ...
```

</details>

**Think.** Your friend's question is about the price. What do the four outputs tell you about the column `price`? And is there anything else in `describe()` that surprises you? (Look at the row `max`.)

*Your answer:*

## Task 2. Prices, and prices that are missing

A model, and a plot, needs numbers. So first you turn the prices into numbers. Then you face the next problem: a large part of them is missing.

**Your turn.** Turn the column `price` into numbers: remove the `$` sign, and anything else that stops Python from reading a number, and convert the result with `.astype(float)`. Store it in the same column.

In [ ]:
# Remove the # at the start of the next line, and replace the dots.
# listings["price"] = ...

<details><summary><b>Hint</b> (click to unfold)</summary>

Two new commands:

- `listings["price"].str.replace("$", "")` replaces a piece of text by another piece, here by nothing, in every value.
- `.astype(float)` turns text such as `"97.00"` into the number 97.0.

You can chain them: `listings["price"].str.replace("$", "").astype(float)`. If that gives an error, read its last line: which value could Python not read? Prices of 1,000 euros or more are written with a comma.

</details>

In [ ]:
# Check
check_2_1()

Now the missing prices. Why would a listing have no price? Inside Airbnb collects the price that the Airbnb website shows, and the website only shows a price for nights that can be booked.

**Your turn.** Count the listings without a price. Then find out which share of *those* listings has `availability_365` equal to 0: they cannot be booked on any day of the coming year.

In [ ]:
n_no_price = ...      # a number
share_closed = ...    # a share between 0 and 1

<details><summary><b>Hint</b> (click to unfold)</summary>

- `listings["price"].isna()` gives `True` for every listing without a price; `.sum()` counts them.
- `listings.loc[listings["price"].isna(), "availability_365"]` gives the availability of only those listings. Compare it with 0, and `.mean()` gives the share of `True`.

</details>

In [ ]:
# Check
check_2_2()

**Example.** Are the missing prices spread evenly over the room types? Run the cell.

In [ ]:
listings["price"].isna().groupby(listings["room_type"]).mean()   # share without a price, per room type

**Think.** Half of the whole homes have no price, but only one in six private rooms. In Amsterdam, a whole home may be rented out to tourists for at most 30 nights a year (the rules for renting out a room in your own home are different). What could explain the missing prices? And suppose you simply drop the listings without a price: for which homes would your answer to your friend then be less reliable?

*Your answer:*

**Your turn.** Your friend asks: *what does a night cost on average?* Compute the mean and the median price. Missing values are skipped automatically.

In [ ]:
mean_price = ...
median_price = ...

In [ ]:
# Check
check_2_3()

**Example.** Models cannot handle missing values, so a common quick fix is *imputation*: fill each gap with one value, for example the median. Lecture 7 treats this in depth. Run the cell and see what the quick fix does to the distribution of the prices.

In [ ]:
filled = listings["price"].fillna(listings["price"].median())   # fill the gaps with the median

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
axes[0].hist(listings["price"], bins=60, range=(0, 1000))
axes[0].set_title("without the missing prices")
axes[1].hist(filled, bins=60, range=(0, 1000))
axes[1].set_title("missing prices filled with the median")
for ax in axes:
    ax.set_xlabel("price per night (euros)")
axes[0].set_ylabel("listings")
plt.show()

**Think.** Which number would you give your friend, the mean or the median, and why? And what did the filling do? Remember "Suspiciously round numbers" in Lecture 2: what would someone conclude from the plot on the right, who does not know it was filled?

*Your answer:*

## Task 3. Listings that stand out

An *outlier* is a value far away from the others. Some outliers are errors, some are real and rare, and you cannot tell which without looking. In this task you find the outliers among the prices, and you ask for each one: what is the most likely reason? Remember the reasons you saw in Lecture 2: a typing error, a different unit or definition, a wrong data type, a default value, or something really unusual.

From now on, you work with the listings that have a price. (Did the conversion of Task 2 not work for you? Then first run `listings = listings_with_prices()` in a new cell: it gives you the table with the prices as numbers.)

In [ ]:
priced = listings[listings["price"].notna()]
len(priced)

**Your turn.** Draw a histogram of all prices in `priced`, with `bins=100`. Label both axes.

In [ ]:
fig, ax = plt.subplots()
...
plt.show()

In [ ]:
# Check
check_3_1()

A histogram shows one distribution. To compare several distributions, put *box plots* side by side, one per group. Each box holds the middle half of the values, the line inside is the median, the whiskers reach to the usual values, and every value beyond the whiskers is drawn as a separate dot: the listings that stand out. (Lecture 2's checklist recommends box plots for outliers; the code is new.)

**Your turn.** Draw one box plot per room type, side by side. Because a few prices are so high, also make the y-axis logarithmic: each step on the axis is then a factor 10 (€10, €100, €1,000, …), so that cheap and expensive listings both get room.

In [ ]:
room_types = ["Shared room", "Private room", "Hotel room", "Entire home/apt"]
groups = [... for room in room_types]    # one list of prices per room type

fig, ax = plt.subplots(figsize=(8, 4))
...
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

- The prices of one room type: `priced.loc[priced["room_type"] == room, "price"]`. Inside the brackets of `groups`, this gives one list per room type.
- `ax.boxplot(groups, tick_labels=room_types)` draws the boxes. (In older versions of matplotlib, the argument is called `labels`.)
- `ax.set_yscale("log")` makes the axis logarithmic.

</details>

In [ ]:
# Check
check_3_2()

**Your turn.** Now look at the listings behind the highest dots. Show the six most expensive listings, with only the columns that help you judge them: `price`, `property_type`, `accommodates`, `bedrooms`, `neighbourhood`, `number_of_reviews` and `availability_365`.

In [ ]:
cols = ["price", "property_type", "accommodates", "bedrooms", "neighbourhood",
        "number_of_reviews", "availability_365"]
...

<details><summary><b>Hint</b> (click to unfold)</summary>

Sort with `priced.sort_values("price", ascending=False)`, then select the columns with `[cols]`, and keep the first rows with `.head(6)`.

</details>

**Think.** Take the top four, one by one. Which ones do you believe, and which not? Give the most likely reason for each. Would you delete them before you answer your friend?

*Your answer:*

Outliers do not only hide in the prices. In Task 1, `describe()` showed that one listing allows a stay of 2,147,483,647 nights. Pile-ups at one value are just as suspicious.

**Your turn.** Look at the most common values of `maximum_nights` with `.value_counts().head()`. Then store how many listings allow exactly 1,125 nights, and the largest value of the column.

In [ ]:
...
n_default = ...
max_nights = ...

In [ ]:
# Check
check_3_3()

**Think.** 365 nights is one year, but what is special about 1,125 nights, and about 2,147,483,647? (Hint: compute $2^{31} - 1$ in a cell.) What would a model do with these values if you leave them as they are?

*Your answer:*

## Task 4. A map of the city, in three colormaps

Location matters for the price, so let us draw a map: longitude on the x-axis, latitude on the y-axis, one dot per listing. With color you can show a third variable. Lecture 2 showed three kinds of colormaps, each for a different kind of data:

- **sequential** (such as `viridis`): amounts, from little to much;
- **diverging** (such as `RdBu`): differences above and below a midpoint;
- **qualitative** (the default colors of matplotlib, `tab10`): groups without an order.

**Example.** Here is the map, colored by price. Run the cell and look at the colors.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
points = ax.scatter(priced["longitude"], priced["latitude"], c=priced["price"],
                    cmap="viridis", s=3)
fig.colorbar(points, ax=ax, label="price per night (euros)")
ax.set_aspect(1.6)      # one degree of longitude is shorter than one of latitude here
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
plt.show()

Almost every dot has the same dark color. The colorbar shows why: it runs to €11,681, so a single listing decides the colors of all the others. The argument `vmax=600` fixes the top of the scale at €600; more expensive listings simply get the top color.

**Your turn.** Draw three maps side by side, one for each kind of colormap:

1. the price, with a sequential colormap and `vmax=600`;
2. the difference between each price and the median price, with a diverging colormap. Choose `vmin` and `vmax` equally far from 0, for example −300 and 300, so that the median price gets the middle color. With `"RdBu_r"`, expensive listings are red and cheap ones blue;
3. the room type, with the default qualitative colors: draw each room type separately, in a loop, with a `label`, and add a legend.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")

# 1. price, sequential
...

# 2. difference with the median price, diverging
diff = priced["price"] - priced["price"].median()
...

# 3. room type, qualitative
for room in priced["room_type"].value_counts().index:   # most common first, rare ones on top
    part = priced[priced["room_type"] == room]
    ...

for ax in axes:
    ax.set_aspect(1.6)
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

- Maps 1 and 2 work like the example: `axes[0].scatter(x, y, c=values, cmap=..., vmax=..., s=3)`, then `fig.colorbar(points, ax=axes[0])`.
- Map 3: inside the loop, `axes[2].scatter(part["longitude"], part["latitude"], s=3, label=room)`. Every call gets the next default color. After the loop: `axes[2].legend(markerscale=4)` (bigger dots in the legend).

</details>

In [ ]:
# Check
check_4()

**Think.** (a) Which map shows best where the expensive listings are, map 1 or map 2? What does white mean in map 2? (b) Lecture 2 warned against the rainbow colormap `jet`. Which of the three maps would suffer most from it, and why? (c) Where are the private rooms, compared with the whole homes?

*Your answer:*

## Task 5. What goes with the price?

Your friend's apartment has a location, a size, a rating, a host. Which of these features go together with the price? In this task you compare groups, rank the features, draw your first regression line, and meet a hidden variable.

**Your turn.** Draw the median price per neighborhood as a sorted horizontal bar chart (Lecture 2, "Bar chart: one number per group"). Why the median and not the mean? You saw the answer in Task 3.

In [ ]:
medians = ...

fig, ax = plt.subplots(figsize=(7, 6))
...
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

`priced.groupby("neighbourhood")["price"].median()` gives one number per neighborhood; add `.sort_values()`. Draw it with `ax.barh(medians.index, medians.values)`.

</details>

In [ ]:
# Check
check_5_1()

**Many features.** The table has eleven numeric features besides the price, and you could draw a scatter plot of each against the price. But real datasets can have thousands of features: in biology, a dataset of gene activity has one column per gene, about 20,000. No one looks at 20,000 scatter plots. A common first step is to *rank* the features by their correlation with the target, and to look closely at the top of the list only.

**Your turn.** Compute the correlation of every feature in `features` with the price, sort it, and draw it as horizontal bars. Store the name of the feature with the strongest correlation in `top_feature`.

In [ ]:
features = ["accommodates", "bedrooms", "beds", "minimum_nights", "maximum_nights",
            "availability_365", "number_of_reviews", "review_scores_rating",
            "host_listings", "latitude", "longitude"]
ranking = ...
top_feature = ...

fig, ax = plt.subplots()
...
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

New: `priced[features].corrwith(priced["price"])` gives the correlation of each column with the price. Then `.sort_values()`, and `ranking.abs().idxmax()` gives the name of the largest value, ignoring the sign. (You can also read the name from the plot and type it in quotes.)

</details>

In [ ]:
# Check
check_5_2()

**Think.** The ranking is a useful first step, but it can mislead you in two ways. (a) `accommodates`, `bedrooms` and `beds` are all near the top. Compute their correlations with each other: `priced[["accommodates", "bedrooms", "beds"]].corr()`. Do you need all three? (b) `latitude` and `longitude` are close to 0. Does that mean location does not matter? Look at your maps of Task 4, and remember Anscombe's quartet.

*Your answer:*

**A first regression line.** A regression line summarizes how the price changes with one feature: its *slope* says how many euros more per extra unit of the feature. Lecture 4 shows how such a line is found; today you only draw it, to see what outliers do to it.

**Example.** The line of price against the number of guests. `np.polyfit(x, y, 1)` finds the best straight line (degree 1) through the points, and gives its slope and its intercept.

In [ ]:
x, y = priced["accommodates"], priced["price"]
slope, intercept = np.polyfit(x, y, 1)

fig, ax = plt.subplots()
ax.scatter(x, y, alpha=0.1, s=10)
guests = np.array([1, 16])
ax.plot(guests, intercept + slope * guests, color="red", label=f"{slope:.0f} euros per guest")
ax.set_xlabel("maximum number of guests")
ax.set_ylabel("price per night (euros)")
ax.legend()
plt.show()

**Your turn.** Fit the line twice: once on all listings in `priced` (as above), and once without the listings of €2,000 or more. Store both slopes. If you like, draw both lines in one plot, and zoom in with `ax.set_ylim(0, 2000)`.

In [ ]:
slope_all = ...

trimmed = ...
slope_trim = ...

In [ ]:
# Check
check_5_3()

**Think.** Ten listings out of more than 6,000 changed the slope by about 16 euros per guest. Why do so few points have so much influence? And which of the two lines would you use for your friend's apartment for 2 guests?

*Your answer:*

**A hidden variable.** Superhosts are experienced, highly rated hosts. Do they ask more, or less?

**Your turn.** Compute the median price per value of `host_is_superhost` (`t` or `f`), and then the median price per room type *and* superhost value.

In [ ]:
by_host = ...
by_both = ...

<details><summary><b>Hint</b> (click to unfold)</summary>

`groupby` also accepts a list of columns: `priced.groupby(["room_type", "host_is_superhost"])["price"].median()` gives one median per combination.

</details>

In [ ]:
# Check
check_5_4()

**Think.** Overall, superhosts ask about €70 less. Within a room type, the difference is much smaller, and for private rooms it even turns around. How can both be true? To find out, compute which share of the listings of superhosts and of the other hosts is a private room: `pd.crosstab(priced["host_is_superhost"], priced["room_type"], normalize="index")`.

*Your answer:*

## Task 6. Sixteen years of reviews

Guests write a review after their stay, so the number of reviews tells you roughly how many guests Airbnb had in Amsterdam. The second file counts the reviews per day.

**Example.** Load the file, and add up the days to months. `resample("MS")` groups the dates by month (`MS`: month start), just like `groupby` groups by a column.

In [ ]:
reviews = pd.read_csv(REVIEWS, parse_dates=["date"])    # parse_dates: read the column as dates
monthly = reviews.set_index("date")["reviews"].resample("MS").sum()
monthly.tail()

**Your turn.** Draw the number of reviews per month as a line plot (Lecture 2, "Line plot: change over time"). Label both axes.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
...
plt.show()

In [ ]:
# Check
check_6()

**Think.** Describe three things you see. For each, is it something real about Amsterdam, or something about how the data were collected? Look closely at the last point.

*Your answer:*

## Task 7. Your own question

Now you decide. Ask the data **one** question, and answer it with **one** plot. A reader should understand the plot without seeing your code (Lecture 2, "From default to readable" and "Show the data"), so:

- label both axes, with units;
- give the plot a title that states your finding, not just the topic;
- leave out decoration that does not carry data.

Choose the chart that fits your question. Some questions to start from, but you are welcome to ask your own:

- Are listings with more reviews rated higher? *(scatter plot)*
- How are the ratings distributed? *(histogram: look at where most ratings are)*
- Which neighborhoods are cheapest *per guest*?
- How many listings belong to hosts who have more than one listing?
- Did the number of new listings per year change? *(the year of `first_review`)*

In [ ]:
# My question: ...

fig, ax = plt.subplots()
...
plt.show()

<details><summary><b>Hint</b> (click to unfold)</summary>

- A ratio per group: `(priced["price"] / priced["accommodates"]).groupby(priced["neighbourhood"]).median()`.
- The year of a date column (new): `pd.to_datetime(listings["first_review"]).dt.year`.

</details>

In [ ]:
# Check
check_7()

**Think.** Write down your finding in one sentence. Then show your plot to your neighbor: can they tell your finding without any explanation from you? They get ten seconds. So does every reader of your plots after this course. At the end of the lab we look at a few plots together.

*Your answer:*

## Check your understanding

Every lecture ends with a few exam-style questions, and so does this lab. Answer each question for yourself first, and only then unfold the answer. We go through the questions together at the end of the lab.

**1.** Which chart fits each question best: a line plot, a scatter plot, a histogram, a bar chart, or box plots side by side?

(a) How did the number of Airbnb guests in Amsterdam change since 2010?<br>
(b) Do listings for more guests cost more?<br>
(c) Which neighborhood has the highest median price?<br>
(d) How are the ratings distributed?<br>
(e) How do the prices of the four room types compare, including the listings that stand out?

<details><summary><b>Answer</b> (click to unfold)</summary>

(a) Line plot: change over time. (b) Scatter plot: the relation between two numbers. (c) Bar chart, sorted: one number per group. (d) Histogram: the distribution of one number. (e) Box plots side by side: several distributions at once, with the outliers as dots.

</details>

**2.** A table of sensor readings has a column `temperature` in °C. `describe()` shows a mean of 412 and a maximum of 9999. What is the most likely explanation, and what do you do first?

A. A heat wave: keep the values &nbsp; B. 9999 is a code for a missing reading: replace it with `np.nan` &nbsp; C. Some readings are in Fahrenheit &nbsp; D. Duplicate rows

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** 9999 is a typical "no value" code, like 2,147,483,647 in `maximum_nights` (Task 3). It is not a temperature, but every computation treats it as one: here it pushes the mean to 412. Replace it with `np.nan` before you compute or plot anything. C would give values around 100, not exactly 9999.

</details>

**3.** One in a hundred bank transactions is fraud. A model labels *every* transaction as "not fraud". Which share of the transactions does it label correctly?

A. 1% &nbsp; B. 50% &nbsp; C. 99% &nbsp; D. It cannot be computed

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** 99 of every 100 transactions are not fraud, so the model is right 99% of the time, and yet it finds no fraud at all. With rare cases, a high share of correct answers says little. And the rare cases are the outliers that matter: removing "strange" transactions before training would remove exactly what the model must find.

</details>

**4.** Over the months of the year, the sales of ice cream and the number of people with sunburn are strongly correlated. What is the most likely explanation?

A. Ice cream causes sunburn &nbsp; B. Sunburn makes people buy ice cream &nbsp; C. A third variable, the summer, drives both &nbsp; D. Pure coincidence

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** Sunny months bring both ice cream and sunburn: a hidden (latent) variable behind the correlation, like the room type behind the superhost prices in Task 5. Correlation is not causation (Lecture 2).

</details>

**5.** Which kind of colormap fits each map: sequential, diverging or qualitative?

(a) The number of listings per neighborhood.<br>
(b) The change in the number of listings since last year, which can be negative or positive.<br>
(c) The neighborhood each listing belongs to.

<details><summary><b>Answer</b> (click to unfold)</summary>

(a) Sequential: amounts, from few to many. (b) Diverging: deviations from a midpoint (no change), with `vmin` and `vmax` equally far from 0. (c) Qualitative: groups without an order. (Lecture 2, "Three kinds of colormaps"; Task 4.)

</details>

**6.** The correlation between a feature and the target is 0.01. Then:

A. The feature is useless for predicting the target &nbsp; B. There is no straight-line relation, but there can be another relation &nbsp; C. The feature has many missing values &nbsp; D. The feature is an outlier

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** A correlation only measures a straight-line relation. Latitude correlates almost 0 with the price, but the map shows that location matters a lot: expensive in the middle, cheaper towards the edges (Task 5, and Anscombe's quartet).

</details>

## If you finish early

Done before the end of the lab? Here are three more tasks. You can also do them at home and compare your work with the solutions on Canvas.

**A. Who did not answer?** Draw the number of missing values per column of `listings` as sorted horizontal bars, showing only the columns with at least one missing value. Three columns have exactly the same number of missing values. Which three, and why?

In [ ]:
...

**B. Default or choice?** In `minimum_nights`, which values are the most common? And how many listings require a stay of 30 nights or more? Who might rent out a home this way, and why would Inside Airbnb's price per night mean something different for them?

In [ ]:
...

**C. One plot for the latent variable.** Show the superhost puzzle of Task 5 in one picture: for each room type, two bars side by side, superhosts and the other hosts. (Hint: `by_both.unstack()` turns the result into a small table, and `table.plot.bar(ax=ax)` draws grouped bars.) Then add a second plot next to it with the share of private rooms for both groups. Does the pair of plots tell the story without words?

In [ ]:
...

## Where this is going

> Plot the data before modeling it. Plots reveal problems that summary numbers hide, and a few lines of `matplotlib` are enough.

Today you found text that should be numbers, missing values with a reason, defaults and codes that are not measurements, outliers that pull a line, a feature that matters without correlating, and a hidden variable. None of these is a mistake of Inside Airbnb: this is what real data look like. No model repairs them for you: garbage in, garbage out.

**Coming up.** Before the next lab there are two lectures:

- Lecture 3 (Monday, November 2): supervised learning, your first model ($k$-nearest neighbors), and how to test a model on new data.
- Lecture 4 (Tuesday, November 3): linear models for regression: how the line of Task 5 is found.

Laptop lab 2 (Thursday, November 5) practices both lectures.

**Reading.** Müller and Guido (2016), Chapter 1: "Essential Libraries and Tools" and "First Things First: Look at Your Data".

---
*This lab practices Lecture 2 of Ş. İlker Birbil. Data: [Inside Airbnb](https://insideairbnb.com), Amsterdam, collected on September 16, 2026, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); we kept 20 columns of the listings and counted the reviews per day.*